In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

print("Libraries imported successfully!")
print("TensorFlow Version:", tf.__version__)

In [ ]:
data = pd.read_csv(
    r"C:\Users\Adwaith\Downloads\english_hindi.csv",
    encoding="utf-8-sig"
)

print("Dataset loaded successfully!")
print(data.head())

print("\nDataset Shape:", data.shape)
print("\nDataset Columns:", data.columns.tolist())

In [ ]:
data = data[["english", "hindi"]].dropna()

data["english"] = data["english"].astype(str).str.strip()
data["hindi"] = data["hindi"].astype(str).str.strip()

data = data[
    (data["english"] != "") &
    (data["hindi"] != "")
].drop_duplicates()

print("Dataset cleaned successfully!")
print("Total sentence pairs:", len(data))
print(data.head())

In [ ]:
data["hindi"] = data["hindi"].apply(
    lambda x: "startseq " + x + " endseq"
)

english_sentences = data["english"].tolist()
hindi_sentences = data["hindi"].tolist()

print("Start and end tokens added!")
print("\nEnglish:", english_sentences[0])
print("Hindi:", hindi_sentences[0])

In [ ]:
translation_dictionary = {}

for english, hindi in zip(
    data["english"],
    data["hindi"]
):

    hindi = hindi.replace("startseq ", "")
    hindi = hindi.replace(" endseq", "")

    translation_dictionary[
        english.lower().strip()
    ] = hindi

print("Translation dictionary created!")
print("Total translations:", len(translation_dictionary))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    english_sentences,
    hindi_sentences,
    test_size=0.2,
    random_state=42
)

print("Dataset split successfully!")

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
eng_tokenizer = Tokenizer(oov_token="<OOV>")

eng_tokenizer.fit_on_texts(X_train)

encoder_input = eng_tokenizer.texts_to_sequences(X_train)

encoder_test = eng_tokenizer.texts_to_sequences(X_test)

print("English tokenization completed!")

print("English Vocabulary Size:",
      len(eng_tokenizer.word_index) + 1)

print("\nOriginal Sentence:", X_train[0])
print("Tokenized Sentence:", encoder_input[0])

In [ ]:
hin_tokenizer = Tokenizer(oov_token="<OOV>")

hin_tokenizer.fit_on_texts(y_train)

decoder_input = hin_tokenizer.texts_to_sequences(y_train)

decoder_test = hin_tokenizer.texts_to_sequences(y_test)

print("Hindi tokenization completed!")

print("Hindi Vocabulary Size:",
      len(hin_tokenizer.word_index) + 1)

print("\nOriginal Sentence:", y_train[0])
print("Tokenized Sentence:", decoder_input[0])

In [ ]:
decoder_target = [
    seq[1:] for seq in decoder_input
]

decoder_target_test = [
    seq[1:] for seq in decoder_test
]

print("Decoder targets created successfully!")

print("Decoder Input:", decoder_input[0])
print("Decoder Target:", decoder_target[0])

In [ ]:
max_eng_len = max(len(seq) for seq in encoder_input)

max_hin_len = max(len(seq) for seq in decoder_input)

encoder_input = pad_sequences(
    encoder_input,
    maxlen=max_eng_len,
    padding="post"
)

encoder_test = pad_sequences(
    encoder_test,
    maxlen=max_eng_len,
    padding="post"
)

decoder_input = pad_sequences(
    decoder_input,
    maxlen=max_hin_len,
    padding="post"
)

decoder_test = pad_sequences(
    decoder_test,
    maxlen=max_hin_len,
    padding="post"
)

decoder_target = pad_sequences(
    decoder_target,
    maxlen=max_hin_len,
    padding="post"
)

decoder_target_test = pad_sequences(
    decoder_target_test,
    maxlen=max_hin_len,
    padding="post"
)

print("Padding completed!")

print("Encoder Input Shape:", encoder_input.shape)
print("Decoder Input Shape:", decoder_input.shape)
print("Decoder Target Shape:", decoder_target.shape)

In [ ]:
eng_vocab_size = len(eng_tokenizer.word_index) + 1

hin_vocab_size = len(hin_tokenizer.word_index) + 1

print("English Vocabulary Size:", eng_vocab_size)
print("Hindi Vocabulary Size:", hin_vocab_size)

latent_dim = 128

print("Latent Dimension:", latent_dim)

In [ ]:
latent_dim = 128

# Encoder
encoder_inputs = Input(shape=(max_eng_len,))

encoder_embedding = Embedding(
    eng_vocab_size,
    latent_dim,
    mask_zero=True
)(encoder_inputs)

encoder_lstm = LSTM(
    latent_dim,
    return_state=True
)

_, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]


# Decoder
decoder_inputs = Input(shape=(max_hin_len,))

decoder_embedding_layer = Embedding(
    hin_vocab_size,
    latent_dim,
    mask_zero=True
)

decoder_embedded = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedded,
    initial_state=encoder_states
)

decoder_dense = Dense(
    hin_vocab_size,
    activation="softmax"
)

decoder_outputs = decoder_dense(decoder_outputs)


# Complete model
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Encoder-Decoder model created successfully!")

model.summary()

In [ ]:
print("\nTraining the Encoder-Decoder model...\n")

history = model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    validation_split=0.1,
    epochs=30,
    batch_size=4,
    verbose=1
)

print("\nModel training completed!")

In [ ]:
loss, accuracy = model.evaluate(
    [encoder_test, decoder_test],
    decoder_target_test,
    verbose=0
)

print("\nModel Evaluation Results")

print("Test Loss:", round(float(loss), 4))

print(
    "Test Token Accuracy:",
    round(float(accuracy) * 100, 2),
    "%"
)

In [ ]:
# Encoder inference model
encoder_model = Model(
    encoder_inputs,
    encoder_states
)

In [ ]:
# Decoder inference inputs
state_input_h = Input(shape=(latent_dim,))
state_input_c = Input(shape=(latent_dim,))

decoder_states_inputs = [
    state_input_h,
    state_input_c
]

decoder_single_input = Input(shape=(1,))

decoder_single_embedded = decoder_embedding_layer(
    decoder_single_input
)

decoder_single_output, h_new, c_new = decoder_lstm(
    decoder_single_embedded,
    initial_state=decoder_states_inputs
)

decoder_single_output = decoder_dense(
    decoder_single_output
)

# Decoder inference model
decoder_model = Model(
    [decoder_single_input] + decoder_states_inputs,
    [decoder_single_output, h_new, c_new]
)

print("Encoder and Decoder inference models created!")

In [ ]:
# Reverse Hindi vocabulary
reverse_hin_index = {
    index: word
    for word, index in hin_tokenizer.word_index.items()
}

start_token = hin_tokenizer.word_index["startseq"]

end_token = hin_tokenizer.word_index["endseq"]


# Translation function
def translate_sentence(sentence):

    sentence = sentence.lower().strip()

    # First check the dataset dictionary
    if sentence in translation_dictionary:

        return translation_dictionary[sentence]

    # Otherwise use the trained LSTM model
    sequence = eng_tokenizer.texts_to_sequences([sentence])

    sequence = pad_sequences(
        sequence,
        maxlen=max_eng_len,
        padding="post"
    )

    states = encoder_model.predict(
        sequence,
        verbose=0
    )

    target_seq = np.array([[start_token]])

    translated_words = []

    for _ in range(max_hin_len):

        output, h, c = decoder_model.predict(
            [target_seq] + states,
            verbose=0
        )

        predicted_id = int(
            np.argmax(output[0, -1, :])
        )

        if predicted_id == 0 or predicted_id == end_token:
            break

        word = reverse_hin_index.get(
            predicted_id, ""
        )

        if word and word not in ("startseq", "<OOV>"):
            translated_words.append(word)

        target_seq = np.array([[predicted_id]])

        states = [h, c]

    return (
        " ".join(translated_words)
        if translated_words
        else "Translation could not be generated."
    )

In [ ]:
print("\n====================================")
print(" ENGLISH TO HINDI TRANSLATOR")
print("====================================")

print("Enter 'exit' to stop the program.")

while True:

    user_sentence = input(
        "\nEnter an English sentence: "
    ).strip()

    if user_sentence.lower() == "exit":

        print("Translation program stopped.")

        break

    if not user_sentence:

        print("Please enter a sentence.")
        continue

    hindi_translation = translate_sentence(
        user_sentence
    )

    print("Hindi Translation:", hindi_translation)


# Save trained model
model.save(
    r"C:\Users\Adwaith\Downloads\english_hindi_model.keras"
)

print("Model saved successfully!")